# SAM2 whole-plant prompting experiment

**Target:** the entire visible target pea plant instance: identifiable leaves, stems, petioles, tendrils, and connected above-ground structure. Exclude neighbouring plants, greenhouse structure, trellis/netting, irrigation, substrate, and unrelated objects. A result that recovers only a leaf is a failure of the defined whole-plant target, not a redefinition of the target.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate/'src').is_dir() and (candidate/'data'/'raw'/'first_sample').is_dir(): return candidate
    raise FileNotFoundError('Open from the thesis project or its notebooks folder.')
ROOT=find_project_root(Path.cwd().resolve()); sys.path.insert(0,str(ROOT))
RESULTS=ROOT/'results'/'sam2_whole_plant'


## Experimental boundary

This Phase 4B experiment uses the same five Phase 3–4 images and pretrained SAM 2.1 Hiera Tiny. It performs no training, fine-tuning, annotation, IoU, Dice, precision, or recall. The SAM2 internal score is **not** validated segmentation accuracy: a high score can still describe a biologically inadequate local region.

In [ ]:
import json, pandas as pd
metadata=json.loads((RESULTS/'run_metadata.json').read_text())
pd.DataFrame(metadata.items(),columns=['Setting','Value'])

## Predeclared prompt plan

Every image receives the same strategy types: whole-plant box; four distributed positives; four positives plus three negatives; box plus positives; and box plus positives and negatives. Coordinates and interaction cost are saved before review to make prompting auditable rather than repeatedly optimized per image.

In [ ]:
prompts=pd.read_csv(RESULTS/'prompts.csv')
prompts[['Image_ID','Filename','Prompt_Strategy','Positive_Point_Count','Negative_Point_Count','Bounding_Box','Point_Coordinates','Human_Interaction','Notes']]

## Run and save

This command saves a four-panel figure and 8-bit binary mask for each image-strategy pair. Masks use 0 for background and 255 for SAM2-predicted foreground. It never writes to raw data.

In [ ]:
import subprocess
subprocess.run([sys.executable,str(ROOT/'scripts'/'run_phase4b_whole_plant.py')],check=True)

## Qualitative evaluation

Complete the qualitative table by visual inspection. Evaluate whole-plant coverage, leaf/stem/tendril coverage, vegetation and trellis leakage, fragmentation, blur response, and interaction cost. Suggested labels are Good/Moderate/Poor; leakage labels are Low/Medium/High. Do not assign values from internal SAM2 scores.

In [ ]:
times=pd.read_csv(RESULTS/'inference_times.csv'); summary=pd.read_csv(RESULTS/'qualitative_summary.csv')
display(times); display(summary)

## Phase 4 versus Phase 4B

Phase 4 used local single-point, box, and limited positive/negative prompts. Phase 4B tests whether points distributed across separate parts of the target plant, optionally combined with a whole-plant box and distractor negatives, improve whole-plant coverage. Better coverage must be balanced against more manual interactions. Record local-only masks, fragmentation, leakage, missed thin structures, trellis-following, and blur failures explicitly.